# L2 — RegEx / Classical Classifier

A GPU-free, interpretable rule engine over the **frozen 86-code label space**.
This is the **hybrid** classifier: it blends two pattern variants and picks a
champion per code.

| Variant | Patterns mined on | Strong on |
|---|---|---|
| **V1** | the full train split, all codes | high-volume head codes with a clear title phrase (606, 537, ...) |
| **V2** | the *overige* bucket (deeds with no large code) | the rare long tail — mining is not drowned out by the head |

For each code the pipeline keeps whichever variant has the higher **out-of-fold
(5-fold CV) F1** on the train split, then applies two deterministic
post-processors: a **koop tie-break** (651/652/696) and a **mutex** pass for
sibling pairs (545/572, 564/585, 606/532). Routing a deed to V2 depends only on
whether **V1 predicts no large code** — never on the true labels — so the
val/test numbers are honest.

All logic lives in `models/regex/` (`pattern_sources`, `patterns`, `scoring`,
`pipeline`); this notebook is a thin runner.

**Inputs** — `data/*.jsonl`, `artifacts/splits/{label_space.json, *_idx.npy}` (run `00_splits.ipynb` once first).
**Outputs** — `artifacts/predictions/regex.parquet` + `regex_val.parquet` (the prediction contract) and `results/regex_metrics.json` (via the shared harness).

## 0. Quick-run switch

`None` = full corpus. An int subsamples the train split for a fast smoke run (val + test stay full).

In [1]:
# None = full train split.  An int = first N train deeds (smoke run only).
SAMPLE_SIZE = None

## 1. Load data, frozen label space, frozen split

The regex never touches the test set during fitting: V1/V2 patterns and thresholds are learned on the **train split**, exactly the split BERT uses.

In [ ]:
import sys
sys.path.insert(0, "..")  # make shared/ and models/ importable from notebooks/

import numpy as np
import pandas as pd

from shared.config import TRAIN_PATH, TEST_PATH
from shared.data    import load_train_test
from shared.labels  import get_mlb
from shared.splits  import load_split
from shared.io      import write_predictions
from shared.harness import evaluate, print_summary

from models.regex import run_hybrid, per_code_f1

train_full_df, test_df = load_train_test(TRAIN_PATH, TEST_PATH)
mlb, classes, num_classes = get_mlb()

train_idx, val_idx = load_split()
train_df = train_full_df.iloc[train_idx].reset_index(drop=True)
val_df   = train_full_df.iloc[val_idx].reset_index(drop=True)

if SAMPLE_SIZE is not None:
    train_df = train_df.head(SAMPLE_SIZE).reset_index(drop=True)
    print(f"[smoke run] train split limited to {len(train_df)} deeds")

print(f"train split: {len(train_df):,}   val: {len(val_df):,}   test: {len(test_df):,}")

## 2. Run the hybrid pipeline

Fits V1 + V2 on the train split, selects the champion per code, predicts val + test, and squashes scores into `[0, 1]` for the contract.

In [ ]:
result = run_hybrid(train_df, val_df, test_df, classes)

print(f"large codes (>= CUTOFF train support): {len(result['large_codes'])}  -> {result['large_codes']}")
print(f"tail codes: {len(result['tail_codes'])}")
print(f"champion: V1 for {result['n_champion_v1']} codes, V2 for {result['n_champion_v2']} codes")
print(f"V1 patterns: {len(result['patterns_v1']):,}   V2 patterns: {len(result['patterns_v2']):,}")
print(f"valuelist candidates accepted at runtime: {len(result['accepted_valuelist'])}")

## 3. Evaluate via the shared harness

One scorer for every method (`zero_division=0`, macro over test-active labels) so micro-F1 / macro-F1 are comparable across regex, BERT, and orchestration.

In [ ]:
y_test = mlb.transform(test_df["rechtsfeitcodes"]).astype(int)
y_val  = mlb.transform(val_df["rechtsfeitcodes"]).astype(int)

m_test = evaluate(y_test, result["test"]["pred"].astype(int), classes, method="regex")
m_val  = evaluate(y_val,  result["val"]["pred"].astype(int),  classes, method="regex_val", save=False)
print_summary([m_test, m_val])

## 4. Write the prediction contract

Long-format parquet, one row per (deed, code), carrying the calibrated score so the orchestration layer can compute margins. `regex` = test, `regex_val` = validation.

In [ ]:
write_predictions(result["test"]["akteId"], result["test"]["scores01"],
                  result["test"]["pred"], classes, method="regex")
write_predictions(result["val"]["akteId"], result["val"]["scores01"],
                  result["val"]["pred"], classes, method="regex_val")

## 5. Per-code test performance

Where the regex earns its place: near-perfect on the standardised head codes, and a handful of tail codes lifted by V2.

In [ ]:
support = y_test.sum(axis=0)
f1 = per_code_f1(y_test, result["test"]["pred"].astype(int))
champ = result["champion"]
per_code = pd.DataFrame({
    "code": [int(c) for c in classes],
    "support_test": support.astype(int),
    "champion": ["V1" if c == 1 else "V2" for c in champ],
    "f1": np.round(f1, 3),
})
per_code = per_code[per_code["support_test"] > 0].sort_values("f1", ascending=False).reset_index(drop=True)
print(f"active codes: {len(per_code)}   |   codes with F1 > 0: {(per_code['f1'] > 0).sum()}")
print("\nTop 12 by F1:")
print(per_code.head(12).to_string(index=False))
print("\nWeakest 12 (still active):")
print(per_code.tail(12).to_string(index=False))

## Notes

- **Why hybrid beats a single regex.** V1 alone is strong on the head but its
  tail mining is swamped by high-volume codes; V2 mines the tail in isolation.
  Champion-per-code keeps the better of the two for every code, so the combined
  classifier is never worse than V1 on any code and recovers several tail codes.
- **Honest evaluation.** Patterns, precision weights, and thresholds are fit on
  the train split only; the test file is scored once at the end.
- **Reproduce.** Run `00_splits.ipynb` once, then run this notebook top to
  bottom with `SAMPLE_SIZE = None`. Results land in `results/regex_metrics.json`
  and `artifacts/predictions/regex{,_val}.parquet`.